# Paper Claim Verification — *How Much Warning Time for Asteroid Impacts Will We Have in the Vera Rubin Era?*

**Paper:** `B12_asteroid.pdf` (Kiker et al., draft 2026-07-06)
**Data:** `data/mar_run_analysis_20260301/` (March run, 30,000 objects) in this repo
**Purpose:** Every exact number or quantitative claim in the paper, recomputed from the archived analysis products, with the steps shown.

Each claim section quotes the paper, computes the value from data, and records a verdict:

| Verdict | Meaning |
|---|---|
| ✅ VERIFIED | Reproduces the paper value (exactly or within rounding) |
| ⚠️ CLOSE | Within a few percent / days, but not exact — likely a cohort or definition difference |
| 📝 WORDING | The number reproduces, but the paper's description of *how* it was computed does not match the code |
| ℹ️ REVIEW | Not a single number — table provided for visual comparison against a figure |

**Cohort definitions** (from `src/adam_impact_study/types.py`):
- `complete()` — `status == "complete"` (29,853 of 30,000; 147 incomplete)
- `discovered()` — non-null `discovery_time` **and** complete
- `observed_but_not_discovered()` — complete, `observations > 0`, null `discovery_time`
- `prograde_orbits()` — inclination < 90°(29,853 → 27,893 after both filters)

**Gotcha:** quivr `Timestamp` columns serialize to parquet as `{days, nanos}` structs — a plain pandas `notna()` is always True. Null-test them via `.mjd()` + `pc.is_null`, or load through `ImpactorResultSummary.from_parquet` as done here.

In [1]:
import sys, collections
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

REPO = "/Users/kathleenkiker/impacts_paper/adam_impacts_study"
DATA = f"{REPO}/data/mar_run_analysis_20260301"
sys.path.insert(0, f"{REPO}/src")

from adam_impact_study.types import ImpactorResultSummary

summary = ImpactorResultSummary.from_parquet(f"{DATA}/summary_results.parquet")
print(f"Loaded summary_results.parquet: {len(summary):,} rows")

diam_km = summary.orbit.diameter.to_numpy(zero_copy_only=False)

# Cohorts (see intro for definitions)
prograde = summary.apply_mask(summary.prograde_orbits())
discovered_all = summary.apply_mask(summary.discovered())
discovered_prog = prograde.apply_mask(prograde.discovered())
print(f"complete: {pc.sum(summary.complete()).as_py():,} | prograde: {len(prograde):,} | "
      f"discovered: {len(discovered_all):,} | discovered+prograde: {len(discovered_prog):,}")

VERDICTS = []
def record(cid, claim, paper, computed, verdict, note=""):
    VERDICTS.append(dict(id=cid, claim=claim, paper=paper, computed=computed,
                         verdict=verdict, note=note))
    print(f"\n[{cid}] {verdict}")
    print(f"  paper:    {paper}")
    print(f"  computed: {computed}")
    if note:
        print(f"  note:     {note}")

Loaded summary_results.parquet: 30,000 rows


complete: 29,853 | prograde: 28,038 | discovered: 15,952 | discovered+prograde: 15,226


## C1 — Synthetic population structure

> *"we simulate a synthetic population of 30,000 Earth-impacting asteroids spanning six diameter bins (40 m, 80 m, 140 m, 250 m, 500 m, and 1 km) and impact dates from 2025 to 2125"* (Abstract)
> *"We generated 500 unique impacting orbits for each combination of ten impact decades (2025–2035, 2035–2045, …, 2115–2125) and six diameter bins"* (§2.1)

In [2]:
oids = summary.orbit.orbit_id.to_pylist()
decades = [o.rsplit("_", 1)[-1] for o in oids]
combo = collections.Counter(zip(decades, diam_km))
impact_t = summary.orbit.impact_time.to_astropy()

print("total objects:   ", len(summary))
print("diameter bins:   ", sorted(set(diam_km)))
print("impact decades:  ", sorted(set(decades)))
print("per-combo counts:", f"min {min(combo.values())}, max {max(combo.values())} "
      f"({len(combo)} combos)")
print("impact dates:    ", impact_t.min().iso[:10], "to", impact_t.max().iso[:10])

ok = (len(summary) == 30_000 and len(set(diam_km)) == 6 and len(set(decades)) == 10
      and min(combo.values()) == max(combo.values()) == 500)
record("C1", "30,000 impactors = 500 x 10 decades x 6 diameter bins, impacts 2025-2125",
       "30,000; 500/combo; 2025-2125",
       f"{len(summary):,}; {min(combo.values())}/combo; "
       f"{impact_t.min().iso[:4]}-{impact_t.max().iso[:4]}",
       "✅ VERIFIED" if ok else "❌ MISMATCH")

total objects:    30000
diameter bins:    [np.float64(0.04), np.float64(0.08), np.float64(0.14), np.float64(0.25), np.float64(0.5), np.float64(1.0)]
impact decades:   ['2025-2035', '2035-2045', '2045-2055', '2055-2065', '2065-2075', '2075-2085', '2085-2095', '2095-2105', '2105-2115', '2115-2125']
per-combo counts: min 500, max 500 (60 combos)
impact dates:     2025-11-03 to 2125-10-28

[C1] ✅ VERIFIED
  paper:    30,000; 500/combo; 2025-2125
  computed: 30,000; 500/combo; 2025-2125


## C2 — 10,000 Monte Carlo variants per IP calculation

> *"adam_core generates 10,000 Monte Carlo orbital variants by sampling from the covariance matrix returned by Find Orb"* (§2.4)

The variant count is visible in the data itself: IP = (impacting variants)/N, so every IP must be an exact multiple of 1/N. (Worth checking because the CLI default is `monte_carlo_samples=1000` — `cli/impact.py:154`.)

In [3]:
ip_col = pq.read_table(f"{DATA}/window_results.parquet",
                       columns=["impact_probability"])["impact_probability"].to_pandas().dropna()
pos = ip_col[ip_col > 0]
frac_1e3 = (np.abs(pos / 1e-3 - np.round(pos / 1e-3)) < 1e-9).mean()
frac_1e4 = (np.abs(pos / 1e-4 - np.round(pos / 1e-4)) < 1e-9).mean()

print(f"windows with an IP: {len(ip_col):,};  min positive IP: {pos.min():g}")
print(f"IPs that are exact multiples of 1e-3: {frac_1e3*100:.1f}%")
print(f"IPs that are exact multiples of 1e-4: {frac_1e4*100:.1f}%")

ok = np.isclose(pos.min(), 1e-4) and frac_1e4 > 0.999
record("C2", "10,000 Monte Carlo variants per nightly IP calculation", "10,000",
       f"IP quantum = {pos.min():g} -> N = {int(round(1/pos.min())):,}",
       "✅ VERIFIED" if ok else "❌ MISMATCH",
       "Data overrides the CLI default of 1,000 — the run config used 10,000")

windows with an IP: 975,447;  min positive IP: 0.0001
IPs that are exact multiples of 1e-3: 67.1%
IPs that are exact multiples of 1e-4: 100.0%

[C2] ✅ VERIFIED
  paper:    10,000
  computed: IP quantum = 0.0001 -> N = 10,000
  note:     Data overrides the CLI default of 1,000 — the run config used 10,000


## C3 — IP computed only with ≥ 3 nights of observations

> *"IP calculations were performed for all (object, night) combinations satisfying a minimum of three nights of observations — the minimum number needed to determine a meaningful preliminary orbit"* (§2.4)

In [4]:
w = pq.read_table(f"{DATA}/window_results.parquet",
                  columns=["observation_nights", "status", "impact_probability"]).to_pandas()
print("all windows: nights min/median/max =",
      w.observation_nights.min(), "/", w.observation_nights.median(), "/", w.observation_nights.max())
with_ip = w[w.impact_probability.notna()]
below3 = (with_ip.observation_nights < 3).sum()
print(f"windows with a computed IP: {len(with_ip):,}; of those with < 3 nights: {below3}")

record("C3", "IP calculations require >= 3 nights of observations", "min 3 nights",
       f"min nights among IP-bearing windows = {with_ip.observation_nights.min()}, "
       f"{below3} below 3",
       "✅ VERIFIED" if below3 == 0 else "❌ MISMATCH")

all windows: nights min/median/max = 3 / 40.0 / 320
windows with a computed IP: 975,447; of those with < 3 nights: 0

[C3] ✅ VERIFIED
  paper:    min 3 nights
  computed: min nights among IP-bearing windows = 3, 0 below 3


## C4 — 70.4% of ≥140 m impactors discovered ("averaged over the realistic distribution of sizes")

> *"Over the full 10-year LSST survey, when averaged over the realistic distribution of sizes, 70.4% of synthetic impactors with diameters ≥140 m were discovered, consistent with the PHA discovery rate of 65.6% projected by Jones et al. (2018)"* (§3.1; also Abstract)

The relevant code is `CompletenessByDiameter.weighted_average()` (`analysis/main.py:597`), whose body is literally `# For now, just the mean`. The per-bin rates come from `calculate_completeness` (`main.py:612`), which counts **raw non-null discovery times over all 30,000 rows** — no `complete`-status or prograde filter (using `discovered()` instead gives 69.8%, not 70.4%).

In [5]:
# raw non-null discovery_time over ALL rows, matching calculate_completeness (main.py:612)
disc_mask = ~pc.is_null(summary.discovery_time.mjd()).to_numpy(zero_copy_only=False)
rates = {d: disc_mask[diam_km == d].mean() * 100 for d in sorted(set(diam_km))}
print(pd.Series(rates, name="discovery %").round(1).rename_axis("diameter (km)"))

big = diam_km >= 0.14
pooled = disc_mask[big].mean() * 100
straight = np.mean([r for d, r in rates.items() if d >= 0.14])
print(f"\npooled >=140m rate:            {pooled:.2f}%")
print(f"straight mean of the 4 bins:   {straight:.2f}%   <- the paper's 70.4%")

# What an actual size-frequency-weighted average would look like:
# differential power-law weights N(>D) ~ D^-alpha across the bins (1 km bin treated as 1-2 km)
edges = np.array([0.14, 0.25, 0.5, 1.0, 2.0])
for alpha in (2.0, 2.35, 2.6):
    wts = edges[:-1] ** -alpha - edges[1:] ** -alpha
    wavg = np.average([rates[d] for d in (0.14, 0.25, 0.5, 1.0)], weights=wts)
    print(f"SFD-weighted average (alpha={alpha}): {wavg:.1f}%")

record("C4", ">=140m discovery rate 70.4%, 'averaged over the realistic distribution of sizes'",
       "70.4%",
       f"{straight:.2f}% — but as a STRAIGHT (unweighted) mean of the 4 bins",
       "📝 WORDING",
       "Number reproduces exactly, but it is NOT weighted by a realistic size distribution "
       "(code comment: 'For now, just the mean'). A realistic SFD weighting is dominated by "
       "the 140m bin (45.1%) and lands near ~48-55%, which would weaken the Jones 65.6% comparison.")

diameter (km)
0.04    12.4
0.08    27.5
0.14    45.1
0.25    63.5
0.50    82.6
1.00    90.6
Name: discovery %, dtype: float64

pooled >=140m rate:            70.43%
straight mean of the 4 bins:   70.43%   <- the paper's 70.4%
SFD-weighted average (alpha=2.0): 52.3%
SFD-weighted average (alpha=2.35): 50.7%
SFD-weighted average (alpha=2.6): 49.8%

[C4] 📝 WORDING
  paper:    70.4%
  computed: 70.43% — but as a STRAIGHT (unweighted) mean of the 4 bins
  note:     Number reproduces exactly, but it is NOT weighted by a realistic size distribution (code comment: 'For now, just the mean'). A realistic SFD weighting is dominated by the 140m bin (45.1%) and lands near ~48-55%, which would weaken the Jones 65.6% comparison.


## C5 — Fewer than 50% of sub-140 m impactors discovered

> *"while fewer than 50% of sub-140 m impactors are discovered"* (Abstract); *"For smaller objects (<140 m), fewer than 50% were discovered"* (§3.1)

In [6]:
small = diam_km < 0.14
small_pooled = disc_mask[small].mean() * 100
print(f"pooled <140m discovery rate: {small_pooled:.1f}%  "
      f"(40m: {rates[0.04]:.1f}%, 80m: {rates[0.08]:.1f}%)")
record("C5", "Fewer than 50% of sub-140m impactors discovered", "<50%",
       f"{small_pooled:.1f}% pooled; worst bin 12.4%, best bin 27.5%",
       "✅ VERIFIED",
       "True but very conservative — the actual pooled rate is ~20%, not just under 50%")

pooled <140m discovery rate: 20.0%  (40m: 12.4%, 80m: 27.5%)

[C5] ✅ VERIFIED
  paper:    <50%
  computed: 20.0% pooled; worst bin 12.4%, best bin 27.5%
  note:     True but very conservative — the actual pooled rate is ~20%, not just under 50%


## C6 — Figure 1 qualitative claims (discovery by 5-year impact window)

> *"1 km objects are discovered at rates approaching or exceeding 90% in most impact windows, while 40 m objects are discovered at rates typically below 20%"* (§3.1)

Note: Figure 1's x-axis stops at 2065–2069, but the simulation extends to 2125 — the grid below covers everything.

In [7]:
iy = np.array([t.datetime.year for t in impact_t])
win = (iy - 2025) // 5
tbl = pd.DataFrame({"window": win, "diam": diam_km, "disc": disc_mask})
grid = tbl.groupby(["window", "diam"])["disc"].mean().unstack() * 100
grid.index = [f"{2025 + 5*w}-{2029 + 5*w}" for w in grid.index]
print(grid.round(1))

km1, m40 = grid[1.0], grid[0.04]
n_win = len(grid)
print(f"\n1km windows >=85%: {(km1 >= 85).sum()}/{n_win} (median {km1.median():.0f}%)")
print(f"40m windows <20%:  {(m40 < 20).sum()}/{n_win} (median {m40.median():.0f}%)")

ok = ((km1 >= 85).mean() > 0.5) and ((m40 < 20).mean() > 0.5)
record("C6", "1km discovered ~>=90% in most windows; 40m typically <20% (Fig 1)",
       "1km ~90% most windows; 40m <20% typical",
       f"1km >=85% in {(km1>=85).sum()}/{n_win} windows; 40m <20% in {(m40<20).sum()}/{n_win}",
       "✅ VERIFIED" if ok else "⚠️ CLOSE")

diam       0.04  0.08  0.14  0.25  0.50  1.00
2025-2029  28.1  42.4  52.9  62.4  69.5  74.8
2030-2034  24.5  38.7  54.8  69.7  83.1  91.6
2035-2039  11.7  28.8  49.3  66.3  87.3  93.7
2040-2044   7.9  26.1  44.7  62.5  83.5  92.4
2045-2049   9.1  23.9  37.9  60.9  81.1  90.9
2050-2054   7.8  21.0  39.9  56.4  80.7  90.9
2055-2059   9.0  21.2  40.8  62.4  86.3  92.5
2060-2064   9.3  22.8  42.1  62.9  83.0  92.3
2065-2069  10.9  26.2  44.1  64.2  83.8  91.7
2070-2074  12.6  27.7  46.7  67.0  84.9  89.8
2075-2079   7.5  23.8  45.8  63.6  82.2  91.6
2080-2084  10.4  24.4  44.4  62.4  82.4  90.8
2085-2089  13.8  28.7  42.9  63.2  83.4  92.7
2090-2094  12.7  29.3  45.3  62.7  80.1  90.6
2095-2099  11.7  29.3  44.4  60.7  81.2  87.9
2100-2104   9.1  25.7  43.1  62.5  83.4  91.3
2105-2109  17.3  29.5  49.6  64.2  86.2  91.7
2110-2114  14.9  26.6  44.8  66.1  81.5  90.3
2115-2119  10.8  28.1  45.8  65.1  85.5  92.8
2120-2124  11.5  25.8  42.3  62.3  80.0  89.2
2125-2129  10.3  37.9  55.2  75.9 

## C7 — 6,102 observed-but-not-discovered objects

> *"Applying this criterion to the full set of 6,102 observed-but-not-discovered objects in our simulation"* (§3.5)

In [8]:
obnd_complete = summary.apply_mask(summary.observed_but_not_discovered())
obnd_prog = prograde.apply_mask(prograde.observed_but_not_discovered())
print(f"complete only:       {len(obnd_complete):,}")
print(f"complete + prograde: {len(obnd_prog):,}")

record("C7", "6,102 observed-but-not-discovered objects", "6,102",
       f"{len(obnd_prog):,} (complete + prograde); {len(obnd_complete):,} without prograde filter",
       "✅ VERIFIED" if len(obnd_prog) == 6102 else "❌ MISMATCH",
       "Exact match REQUIRES the prograde (i<90°) filter, which the paper never mentions")

complete only:       6,532
complete + prograde: 6,102

[C7] ✅ VERIFIED
  paper:    6,102
  computed: 6,102 (complete + prograde); 6,532 without prograde filter
  note:     Exact match REQUIRES the prograde (i<90°) filter, which the paper never mentions


## C8 — Median IP at discovery = 7%

> *"The median IP at the time of discovery is 7%"* (Abstract); *"The median IP at the time of discovery is 7%, meaning that the initial orbit determined from the discovery tracklets already points strongly toward an Earth collision."* (§3.3)

In [9]:
for label, t in [("all discovered", discovered_all), ("prograde discovered", discovered_prog)]:
    ip = t.ip_at_discovery_time.to_numpy(zero_copy_only=False)
    ip = ip[~np.isnan(ip)]
    zero_frac = (ip == 0).mean() * 100
    print(f"{label}: median {np.median(ip)*100:.2f}% | nonzero-only median "
          f"{np.median(ip[ip > 0])*100:.2f}% | IP=0 at discovery: {zero_frac:.0f}% of objects")

record("C8", "Median IP at time of discovery = 7%", "7%",
       "2.9% (all discovered) / 6.5% (excluding IP=0)",
       "⚠️ CLOSE",
       "Only the nonzero-only median rounds toward 7%. A naive recomputation gives ~3%. "
       "The zero-exclusion (or whatever cohort produced 7%) should be stated explicitly.")

all discovered: median 2.77% | nonzero-only median 6.60% | IP=0 at discovery: 12% of objects
prograde discovered: median 2.85% | nonzero-only median 6.47% | IP=0 at discovery: 12% of objects

[C8] ⚠️ CLOSE
  paper:    7%
  computed: 2.9% (all discovered) / 6.5% (excluding IP=0)
  note:     Only the nonzero-only median rounds toward 7%. A naive recomputation gives ~3%. The zero-exclusion (or whatever cohort produced 7%) should be stated explicitly.


## C9 — 83.0% of discovered objects reach IP > 90%

> *"83% of discovered objects ultimately reach an IP exceeding 90%"* (Abstract); *"More precisely, 83.0% of discovered objects reach an IP exceeding 90% before the end of the 10-year survey."* (§3.3)

In [10]:
def frac_reaching(t, col):
    return (~pc.is_null(getattr(t, col).mjd()).to_numpy(zero_copy_only=False)).mean() * 100

r_all = frac_reaching(discovered_all, "ip_threshold_90_percent")
r_prog = frac_reaching(discovered_prog, "ip_threshold_90_percent")
print(f"all discovered: {r_all:.1f}% | prograde discovered: {r_prog:.1f}%")

record("C9", "83.0% of discovered objects reach IP > 90%", "83.0%", f"{r_all:.1f}%",
       "✅ VERIFIED" if round(r_all, 1) == 83.0 else "⚠️ CLOSE")

all discovered: 83.0% | prograde discovered: 82.8%

[C9] ✅ VERIFIED
  paper:    83.0%
  computed: 83.0%


## C10 — 4.5% of discovered objects never exceed 1% IP

> *"A notable subset of 4.5% of discovered objects never exceeds a 1% IP throughout the simulation."* (§3.3)

In [11]:
for label, t in [("all discovered", discovered_all), ("prograde discovered", discovered_prog)]:
    never = pc.is_null(t.ip_threshold_1_percent.mjd()).to_numpy(zero_copy_only=False).mean() * 100
    print(f"{label}: {never:.2f}% never cross 1%")

n_all = pc.is_null(discovered_all.ip_threshold_1_percent.mjd()).to_numpy(zero_copy_only=False).mean() * 100
record("C10", "4.5% of discovered objects never exceed 1% IP", "4.5%", f"{n_all:.1f}%",
       "✅ VERIFIED" if abs(n_all - 4.5) < 0.2 else "⚠️ CLOSE")

all discovered: 4.42% never cross 1%
prograde discovered: 4.49% never cross 1%

[C10] ✅ VERIFIED
  paper:    4.5%
  computed: 4.4%


## C11 — Discovery-to-1%-crossing: median 45 days, mean 326 days

> *"For objects discovered with an IP below 1% that subsequently rise above this threshold, the median time from discovery to crossing 1% is 45 days, while the mean is 326 days."* (§3.3)

The crossing epoch (`ip_threshold_1_percent`) is the **first** window at/above 1% over the whole simulation, so it can precede a late discovery; the variants below differ in how those cases are treated.

In [12]:
for label, t in [("all discovered", discovered_all), ("prograde discovered", discovered_prog)]:
    ipd = np.nan_to_num(t.ip_at_discovery_time.to_numpy(zero_copy_only=False), nan=0.0)
    dt = pc.subtract(t.ip_threshold_1_percent.mjd(), t.discovery_time.mjd()).to_numpy(zero_copy_only=False)
    below = ipd < 0.01
    for vlabel, m in [("crossing strictly after discovery (dt>0)", below & ~np.isnan(dt) & (dt > 0)),
                      ("any crossing (incl. dt<=0)          ", below & ~np.isnan(dt))]:
        print(f"{label:20s} | {vlabel} | n={m.sum():5d} "
              f"median={np.median(dt[m]):5.0f}d  mean={np.mean(dt[m]):5.0f}d")

record("C11", "Discovered <1% then crossing 1%: median 45 d, mean 326 d", "45 d / 326 d",
       "47 d / 316 d (closest variant: all discovered, any crossing) — strict variant 52 d / 342 d",
       "⚠️ CLOSE",
       "No variant reproduces 45/326 exactly; the paper should state how crossings at/before "
       "discovery are handled")

all discovered       | crossing strictly after discovery (dt>0) | n= 5907 median=   52d  mean=  342d
all discovered       | any crossing (incl. dt<=0)           | n= 6147 median=   47d  mean=  316d
prograde discovered  | crossing strictly after discovery (dt>0) | n= 5571 median=   48d  mean=  336d
prograde discovered  | any crossing (incl. dt<=0)           | n= 5806 median=   43d  mean=  309d

[C11] ⚠️ CLOSE
  paper:    45 d / 326 d
  computed: 47 d / 316 d (closest variant: all discovered, any crossing) — strict variant 52 d / 342 d
  note:     No variant reproduces 45/326 exactly; the paper should state how crossings at/before discovery are handled


## C12 — Warning times (Figure 8 and text claims)

> *"for 1 km objects impacting in 2045, the mean warning time reaches 18 years, while for 40 m objects in the same year it is ~12 years. For 140 m objects impacting ~10 years into the survey, the mean warning time is approximately 8 years"* (§3.4)
> *"For a 140 m impactor with a ~10-year time-to-impact, the mean warning time is approximately 8 years."* (Abstract)

Reproduces Figure 8's exact pipeline: cohort = complete + discovered, impact years 2036–2045 (`plots.py:29`), warning time = `max(impact − 1%-crossing, impact − discovery)` in days (`types.py:257`). Note this is subtly different from the paper's definition ("interval from when IP first exceeds 1% to the predicted impact date") whenever discovery happens *after* the 1% crossing.

In [13]:
cd = summary.apply_mask(summary.complete())
cd = cd.apply_mask(cd.discovered())
iy_cd = np.array([t.datetime.year for t in cd.orbit.impact_time.to_astropy()])
wt_yr = cd.warning_time().to_numpy(zero_copy_only=False) / 365.25
d_cd = cd.orbit.diameter.to_numpy(zero_copy_only=False)

rows = {}
for year in range(2036, 2046):
    rows[year] = {d: np.nanmean(wt_yr[(iy_cd == year) & (d_cd == d)])
                  for d in sorted(set(d_cd))}
fig8 = pd.DataFrame(rows).T
fig8.columns = [f"{c*1000:.0f}m" for c in fig8.columns]
print("Mean warning time (years) — compare against Figure 8 bar labels:")
print(fig8.round(1))

c_140_2036, c_1km_2045, c_40_2045 = fig8.loc[2036, "140m"], fig8.loc[2045, "1000m"], fig8.loc[2045, "40m"]
record("C12a", "140m impactor, ~10-yr time-to-impact: mean warning ~8 yr", "~8 yr",
       f"{c_140_2036:.1f} yr (140m, impact year 2036)",
       "✅ VERIFIED" if abs(c_140_2036 - 8) < 0.5 else "⚠️ CLOSE")
record("C12b", "1km impacting 2045: mean warning reaches 18 yr", "18 yr",
       f"{c_1km_2045:.1f} yr",
       "✅ VERIFIED" if abs(c_1km_2045 - 18) < 0.5 else "⚠️ CLOSE")
record("C12c", "40m impacting 2045: mean warning ~12 yr", "~12 yr",
       f"{c_40_2045:.1f} yr",
       "✅ VERIFIED" if abs(c_40_2045 - 12) < 1.0 else "⚠️ CLOSE")

Mean warning time (years) — compare against Figure 8 bar labels:
       40m   80m  140m  250m  500m  1000m
2036   1.7   6.9   8.0   8.4   9.3    9.2
2037   7.5   7.7   8.8   8.8   9.5    9.9
2038   8.3  10.0  10.0  10.3  11.0   11.4
2039  11.2  11.2  11.5  11.4  12.4   12.5
2040  11.8  11.6  11.6  12.5  13.0   13.4
2041  12.3  12.3  12.6  13.1  13.4   14.0
2042  14.3  13.1  13.9  14.1  14.7   15.2
2043  15.6  13.8  15.0  15.1  15.9   16.1
2044  12.8  16.0  15.2  16.9  17.2   17.3
2045  12.7  14.3  15.9  16.9  18.1   18.1

[C12a] ✅ VERIFIED
  paper:    ~8 yr
  computed: 8.0 yr (140m, impact year 2036)

[C12b] ✅ VERIFIED
  paper:    18 yr
  computed: 18.1 yr

[C12c] ✅ VERIFIED
  paper:    ~12 yr
  computed: 12.7 yr


## C13 — Realization time grid (Figure 5 cross-check)

> *"The mean realization time—the time from first detection to when the object's IP permanently exceeds a high-confidence threshold—is shown in Figure 5"* (§3.3)

Two definitional caveats versus the code: the plotted quantity is discovery → **first** crossing of the 0.01% threshold (`compute_ip_threshold_date`, `main.py:317` keeps the *first* window at/above threshold — not "permanently exceeds"), and the clock starts at formal *discovery*, not "first detection". Table below is for visual comparison with Figure 5's bars and n-labels.

In [14]:
rt = pc.subtract(cd.ip_threshold_0_dot_01_percent.mjd(),
                 cd.discovery_time.mjd()).to_numpy(zero_copy_only=False)
dec = (iy_cd // 10) * 10
rt_tbl = pd.DataFrame({"decade": dec, "diam": d_cd, "rt": rt})
mean_rt = rt_tbl.groupby(["decade", "diam"])["rt"].mean().unstack()
n_rt = rt_tbl.dropna(subset=["rt"]).groupby(["decade", "diam"])["rt"].count().unstack()
mean_rt.columns = n_rt.columns = [f"{c*1000:.0f}m" for c in mean_rt.columns]
print("Mean days, discovery -> first 0.01% IP crossing (compare Fig 5 bar heights):")
print(mean_rt.round(1))
print("\nCounts n per cell (compare Fig 5 n= labels):")
print(n_rt)

record("C13", "Realization times by decade/diameter (Fig 5)", "Fig 5 bars + n labels",
       "grids printed above for visual comparison",
       "ℹ️ REVIEW",
       "Code computes discovery -> FIRST 0.01% crossing; paper text says 'first detection' and "
       "'permanently exceeds' — both wordings differ from the implementation")

Mean days, discovery -> first 0.01% IP crossing (compare Fig 5 bar heights):
          40m    80m   140m   250m   500m  1000m
decade                                          
2020    -22.2  -48.9  -72.8  -90.1  -64.0 -101.2
2030   -117.5 -317.1 -339.2 -293.1 -271.1 -185.8
2040   -102.8 -314.4 -324.8 -463.7 -287.1 -204.2
2050    -58.2 -160.0 -342.1 -353.6 -285.7 -220.3
2060    -35.1 -189.7 -244.9 -398.3 -300.9 -249.0
2070    -79.9 -144.6 -276.1 -346.4 -308.2 -267.8
2080   -184.7 -198.6 -233.0 -332.4 -271.4 -247.7
2090   -230.5 -157.3 -257.1 -344.3 -303.6 -313.2
2100   -173.4 -226.7 -327.8 -328.5 -289.3 -231.0
2110    -33.9 -270.3 -252.1 -294.1 -249.6 -217.8
2120     32.9 -246.8 -284.3 -272.5 -332.2 -207.5

Counts n per cell (compare Fig 5 n= labels):
        40m  80m  140m  250m  500m  1000m
decade                                   
2020     59   89   111   131   146    155
2030     88  158   244   316   393    426
2040     45  132   222   325   437    479
2050     41  105   199   294  

## C14 — Generous linking recovers 2,355 (38.5%) of undiscovered objects

> *"we defined a more generous linking criterion: any 6 observations within a 30-day window spanning at least 2 distinct nights. Applying this criterion to the full set of 6,102 observed-but-not-discovered objects in our simulation, we identify 2,355 (38.5%) that could in principle be discovered"* (§3.5); Abstract: *"recover an additional 38.5%"*.

Code criterion (`compute_discovery_dates_optimistic`, `main.py:107`): 6 observations within 30 days whose arc is ≥ 1.0 day — an arc-length proxy for "2 distinct nights", not literally the same thing.

In [15]:
def n_recoverable(t):
    return (~pc.is_null(t.discovery_time_optimistic.mjd()).to_numpy(zero_copy_only=False)).sum()

r_prog = n_recoverable(obnd_prog)
r_comp = n_recoverable(obnd_complete)
print(f"complete+prograde: {r_prog:,}/{len(obnd_prog):,} = {r_prog/len(obnd_prog)*100:.1f}%")
print(f"complete only:     {r_comp:,}/{len(obnd_complete):,} = {r_comp/len(obnd_complete)*100:.1f}%")

record("C14", "2,355 (38.5%) of 6,102 undiscovered recoverable via generous linking",
       "2,355 (38.5%)",
       f"{r_prog:,} ({r_prog/len(obnd_prog)*100:.1f}%) on complete+prograde",
       "⚠️ CLOSE" if r_prog != 2355 else "✅ VERIFIED",
       "Off by 10 objects — possibly one more sub-filter in the paper's count. Also note the "
       "code criterion is arc>=1.0 day, not '2 distinct nights' as worded.")

complete+prograde: 2,365/6,102 = 38.8%
complete only:     2,520/6,532 = 38.6%

[C14] ⚠️ CLOSE
  paper:    2,355 (38.5%)
  computed: 2,365 (38.8%) on complete+prograde
  note:     Off by 10 objects — possibly one more sub-filter in the paper's count. Also note the code criterion is arc>=1.0 day, not '2 distinct nights' as worded.


## C15 — Generous linking gains: median +7 days, mean +240 days warning time

> *"this would increase the median warning time by 7 days and the mean warning time by 240 days"* (§3.5); Abstract: *"increasing mean warning times by up to 240 days"*.

Computed as the discovery-epoch advance (standard discovery time − optimistic discovery time) for already-discovered objects that also satisfy the optimistic criterion.

In [16]:
for label, t in [("all discovered", discovered_all), ("prograde discovered", discovered_prog)]:
    delta = pc.subtract(t.discovery_time.mjd(),
                        t.discovery_time_optimistic.mjd()).to_numpy(zero_copy_only=False)
    dv = delta[~np.isnan(delta)]
    print(f"{label}: n={len(dv):,}  median={np.median(dv):.0f} d  mean={np.mean(dv):.0f} d")

delta_all = pc.subtract(discovered_all.discovery_time.mjd(),
                        discovered_all.discovery_time_optimistic.mjd()).to_numpy(zero_copy_only=False)
dv = delta_all[~np.isnan(delta_all)]
record("C15", "Generous linking: median +7 d, mean +240 d warning time", "+7 d / +240 d",
       f"+{np.median(dv):.0f} d / +{np.mean(dv):.0f} d (all discovered)",
       "✅ VERIFIED" if abs(np.median(dv) - 7) <= 1.5 and abs(np.mean(dv) - 240) <= 5 else "⚠️ CLOSE",
       "Assumes the paper equates warning-time gain with discovery-epoch advance")

all discovered: n=15,952  median=8 d  mean=242 d
prograde discovered: n=15,226  median=8 d  mean=239 d

[C15] ✅ VERIFIED
  paper:    +7 d / +240 d
  computed: +8 d / +242 d (all discovered)
  note:     Assumes the paper equates warning-time gain with discovery-epoch advance


---
# Review-requested checks (comment doc, 2026-07-06)

Numbers the internal review asked to verify or quantify. Items marked 📊 COMPUTED are new
quantities requested for the paper, not existing claims.

## R1 — Discovery fractions above/below 1 AU

> Review: *"There's a couple places we should double check the numbers. Eg. discover fractions over/under 1AU"*

Semi-major axis from the impactor orbits (Keplerian conversion of the initial state). Discovery = raw non-null discovery time (same convention as C4/completeness).

In [17]:
kep = summary.orbit.coordinates.to_keplerian()
a_au = kep.a.to_numpy(zero_copy_only=False)

rows = []
for label, dmask in ([(f"{d*1000:.0f}m", diam_km == d) for d in sorted(set(diam_km))]
                     + [(">=140m", diam_km >= 0.14), ("all", np.ones(len(diam_km), bool))]):
    inner, outer, band = dmask & (a_au < 1), dmask & (a_au >= 1), dmask & (np.abs(a_au - 1) < 0.05)
    rows.append({"bin": label,
                 "a<1AU disc%": disc_mask[inner].mean() * 100, "n(a<1)": inner.sum(),
                 "a>=1AU disc%": disc_mask[outer].mean() * 100, "n(a>=1)": outer.sum(),
                 "|a-1|<0.05 disc%": disc_mask[band].mean() * 100, "n(band)": band.sum()})
r1 = pd.DataFrame(rows).set_index("bin")
print(r1.round(1))

big = diam_km >= 0.14
record("R1", "Discovery fraction below vs above 1 AU (review request)", "(not yet in paper)",
       f">=140m: a<1AU {disc_mask[big & (a_au<1)].mean()*100:.1f}% vs a>=1AU "
       f"{disc_mask[big & (a_au>=1)].mean()*100:.1f}%; synodic band |a-1|<0.05: "
       f"{disc_mask[big & (np.abs(a_au-1)<0.05)].mean()*100:.1f}%",
       "📊 COMPUTED",
       "Interior-Earth-orbit and near-1AU objects are discovered at sharply lower rates, "
       "quantifying the Fig 2 / section 3.2 discussion")

        a<1AU disc%  n(a<1)  a>=1AU disc%  n(a>=1)  |a-1|<0.05 disc%  n(band)
bin                                                                          
40m            13.2     951          12.2     4049              17.2      309
80m            28.0     951          27.4     4049              28.2      309
140m           39.1     951          46.5     4049              38.2      309
250m           54.6     951          65.5     4049              48.2      309
500m           72.0     951          85.1     4049              66.0      309
1000m          79.4     951          93.2     4049              76.7      309
>=140m         61.3    3804          72.6    16196              57.3     1236
all            47.7    5706          55.0    24294              45.7     1854

[R1] 📊 COMPUTED
  paper:    (not yet in paper)
  computed: >=140m: a<1AU 61.3% vs a>=1AU 72.6%; synodic band |a-1|<0.05: 57.3%
  note:     Interior-Earth-orbit and near-1AU objects are discovered at sharply lower rates,

## R2 — What fraction of undiscovered ≥140 m impactors have a < 1 AU? |a − 1| < 0.05 AU?

> Review: *"Can we quantify the a<1AU and high synodic period cases? Ex what fraction of undiscovered ≥140 m impactors have a < 1 AU? What fraction have |a − 1 AU| < 0.05 AU?"*

Undiscovered = complete run, null discovery time. Shown for ≥140 m, split into observed-but-not-discovered vs never-observed, with dynamical classes for context.

In [18]:
complete_np = summary.complete().to_numpy(zero_copy_only=False)
undisc = complete_np & ~disc_mask & (diam_km >= 0.14)
obs_np = summary.observations.to_numpy(zero_copy_only=False).astype(float)
big = diam_km >= 0.14

# Raw counts + fractions for both orbit categories, with the full-population
# baseline and the discovered complement for comparison
groups = {
    "full >=140m population":       big,
    "  discovered":                 big & complete_np & disc_mask,
    "  undiscovered (all)":         undisc,
    "    observed-not-discovered":  undisc & (obs_np > 0),
    "    never observed":           undisc & (obs_np == 0),
    "  incomplete runs (excluded)": big & ~complete_np,
}
rows = []
for label, m in groups.items():
    n = m.sum()
    inner = int((a_au[m] < 1).sum())
    band = int((np.abs(a_au[m] - 1) < 0.05).sum())
    rows.append({"group": label, "n": n,
                 "a<1AU": inner, "a<1AU %": inner / n * 100,
                 "|a-1|<0.05": band, "|a-1|<0.05 %": band / n * 100})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.1f}"))
# discovered + undiscovered + incomplete = full population
n_disc = (big & complete_np & disc_mask).sum()
n_inc = (big & ~complete_np).sum()
assert n_disc + undisc.sum() + n_inc == big.sum()
print(f"\nsum check: {n_disc:,} discovered + {undisc.sum():,} undiscovered + "
      f"{n_inc} incomplete = {big.sum():,}")
print(f"(of the {n_inc} incomplete runs, {(big & ~complete_np & disc_mask).sum()} have a "
      f"discovery time but are excluded from 'discovered' by the status filter)")

n_u = undisc.sum()
fa1 = (a_au[undisc] < 1).mean() * 100
fband = (np.abs(a_au[undisc] - 1) < 0.05).mean() * 100
funion = ((a_au[undisc] < 1) | (np.abs(a_au[undisc] - 1) < 0.05)).mean() * 100
print(f"\nundiscovered union of the two categories: {funion:.1f}%")

dyn = pd.Series(np.array(summary.orbit.dynamical_class.to_pylist())[undisc])
print("\ndynamical classes of undiscovered >=140m:")
print(dyn.value_counts())

record("R2", "Orbit fractions of undiscovered >=140m impactors (review request)", "(not yet in paper)",
       f"a<1AU: {fa1:.1f}% (n={int((a_au[undisc]<1).sum())}); |a-1|<0.05AU: {fband:.1f}% "
       f"(n={int((np.abs(a_au[undisc]-1)<0.05).sum())}); union {funion:.1f}% (of n={n_u:,}); "
       f"population baseline 19.0% / 6.2%",
       "📊 COMPUTED",
       "Never-observed subset: a<1AU 16.2% (325/2,010) but synodic band 14.6% (293) — 2.4x the "
       "population's 6.2% baseline; the band hides objects from the survey entirely")

                       group     n  a<1AU  a<1AU %  |a-1|<0.05  |a-1|<0.05 %
      full >=140m population 20000   3804     19.0        1236           6.2
                  discovered 13960   2321     16.6         705           5.1
          undiscovered (all)  5899   1468     24.9         527           8.9
     observed-not-discovered  3889   1143     29.4         234           6.0
              never observed  2010    325     16.2         293          14.6
  incomplete runs (excluded)   141     15     10.6           4           2.8

sum check: 13,960 discovered + 5,899 undiscovered + 141 incomplete = 20,000
(of the 141 incomplete runs, 126 have a discovery time but are excluded from 'discovered' by the status filter)

undiscovered union of the two categories: 29.6%

dynamical classes of undiscovered >=140m:
APO    4235
ATE    1468
OMB     196
Name: count, dtype: int64

[R2] 📊 COMPUTED
  paper:    (not yet in paper)
  computed: a<1AU: 24.9% (n=1468); |a-1|<0.05AU: 8.9% (n=527); union 2

## R3 — Is the IP gate really "minimum of three nights"?

> Review: *"'IP calculations were performed on minimum of three nights' - is that true, I thought it was by total number of observations"*

Code (`impacts_study.py:349,362,366`): windows are generated starting from the **3rd unique night**, *and* a window is skipped unless it has **≥ 6 total observations**. So the paper's sentence is incomplete — both conditions gate the IP calculation.

In [19]:
wrc = pq.read_table(f"{DATA}/window_results.parquet",
                    columns=["observation_nights", "observation_count", "impact_probability"]).to_pandas()
with_ip = wrc[wrc.impact_probability.notna()]
print(f"IP-bearing windows: {len(with_ip):,}")
print(f"  min nights:       {with_ip.observation_nights.min()}")
print(f"  min observations: {with_ip.observation_count.min()}")
print(f"  windows with exactly 3 nights: {(with_ip.observation_nights==3).sum():,}; "
      f"with exactly 6 obs: {(with_ip.observation_count==6).sum():,}")

record("R3", "IP gate: '>=3 nights' as stated in section 2.4", ">=3 nights",
       f"BOTH >=3 nights AND >=6 observations (data: min nights={with_ip.observation_nights.min()}, "
       f"min obs={with_ip.observation_count.min()})",
       "📝 WORDING",
       "Reviewer is half right: the gate is nights-based AND count-based; paper should state both")

IP-bearing windows: 975,447
  min nights:       3
  min observations: 6
  windows with exactly 3 nights: 7,374; with exactly 6 obs: 8,677

[R3] 📝 WORDING
  paper:    >=3 nights
  computed: BOTH >=3 nights AND >=6 observations (data: min nights=3, min obs=6)
  note:     Reviewer is half right: the gate is nights-based AND count-based; paper should state both


## R4 — Do objects impacting post-2080 really have "progressively fewer observations"?

> Fig 3 caption: *"objects impacting post-2080 have progressively fewer observations available for orbit refinement."*
> Review (Ed): the real cause of declining max IP is longer propagation, not fewer observations — *"we could make a plot of number of observations of each impacting decade to see if it goes down"*

In [20]:
iy_all = np.array([t.datetime.year for t in impact_t])
dec_all = (iy_all // 10) * 10
obs_tbl = pd.DataFrame({"decade": dec_all, "obs": obs_np, "windows":
                        summary.windows.to_numpy(zero_copy_only=False).astype(float),
                        "disc": disc_mask, "diam": diam_km})
g = obs_tbl.groupby("decade").agg(mean_obs=("obs", "mean"), median_obs=("obs", "median"),
                                  mean_windows=("windows", "mean"), n=("obs", "size"))
gd = obs_tbl[obs_tbl.disc].groupby("decade").agg(mean_obs_disc=("obs", "mean"),
                                                 median_obs_disc=("obs", "median"))
print(g.join(gd).round(1))

post2040 = g.loc[g.index >= 2040, "mean_obs"]
trend = np.polyfit(post2040.index, post2040.values, 1)[0]
print(f"\nslope of mean obs per object vs decade (2040+): {trend:+.2f} obs/decade")

decline = post2040.loc[2080:].is_monotonic_decreasing and (post2040.loc[2080] - post2040.iloc[-1]) > 5
record("R4", "Fig 3 caption: post-2080 objects have progressively fewer observations",
       "fewer observations post-2080",
       f"mean obs/object by decade 2040->2120 is essentially flat (slope {trend:+.2f}/decade); "
       f"see table",
       "❌ MISMATCH" if not decline else "✅ VERIFIED",
       "Supports the review comment: declining max IP for late decades is propagation distance, "
       "not observation count (except the 2020s/2030s, which get the final-apparition boost)")

        mean_obs  median_obs  mean_windows     n  mean_obs_disc  \
decade                                                            
2020        30.4        17.0          12.3  1260           51.9   
2030        66.0        29.0          29.6  2796          108.6   
2040        77.8        25.5          35.7  3204          145.1   
2050        73.8        23.0          34.4  2988          140.6   
2060        76.2        25.0          33.6  2928          139.9   
2070        73.4        26.5          33.4  2994          132.3   
2080        77.5        26.0          34.1  2982          141.0   
2090        70.9        26.0          31.7  3090          128.9   
2100        71.6        28.0          32.9  3042          126.9   
2110        78.2        29.0          35.2  2982          139.2   
2120        77.4        26.0          35.6  1734          142.0   

        median_obs_disc  
decade                   
2020               42.0  
2030               80.0  
2040              111.0 

## R5 — Calendar-decade binning imbalance

> Review: *"It is a little confusing that we have two different binnings - the 2025-2035 bin and '2020s' impacting decade. Ex in fig 3 this means the first and last decades have half the orbits of the other bins."*

In [21]:
counts = pd.Series(dec_all).value_counts().sort_index()
print(counts.rename("objects per calendar impact decade"))
record("R5", "First/last calendar decades hold about half the objects of the others",
       "~half in 2020s and 2120s",
       f"2020s: {counts.loc[2020]:,}; 2120s: {counts.loc[2120]:,}; middle decades ~{int(counts.loc[2050]):,}",
       "✅ VERIFIED",
       "Population decade bins (2025-2035 etc.) straddle calendar decades, so Figs 3-6's first "
       "and last x-bins have roughly half the sample size")

2020    1260
2030    2796
2040    3204
2050    2988
2060    2928
2070    2994
2080    2982
2090    3090
2100    3042
2110    2982
2120    1734
Name: objects per calendar impact decade, dtype: int64

[R5] ✅ VERIFIED
  paper:    ~half in 2020s and 2120s
  computed: 2020s: 1,260; 2120s: 1,734; middle decades ~2,988
  note:     Population decade bins (2025-2035 etc.) straddle calendar decades, so Figs 3-6's first and last x-bins have roughly half the sample size


## R6 — Is Fig 5's 0.01% threshold a single Monte Carlo draw?

> Review: *"Fig 5 - are we sure this is supposed to be the .01% IP threshold? That's a single monte carlo draw."*

With 10,000 variants (C2), IP resolution is exactly 1/10,000 = 0.01% — so crossing the 0.01% threshold means **at least one** impacting variant. Below: at the first window that crosses the threshold, how often is the IP exactly one draw?

In [22]:
wr = pq.read_table(f"{DATA}/window_results.parquet",
                   columns=["orbit_id", "observation_end", "impact_probability"])
fc = pd.DataFrame({
    "orbit_id": wr["orbit_id"].to_pandas(),
    "days": pc.struct_field(wr["observation_end"], "days").to_pandas(),
    "nanos": pc.struct_field(wr["observation_end"], "nanos").to_pandas(),
    "ip": wr["impact_probability"].to_pandas(),
})
fc = fc[fc.ip >= 1e-4].sort_values(["orbit_id", "days", "nanos"])
first_cross = fc.groupby("orbit_id").first()
single = (first_cross.ip == 1e-4).mean() * 100
low = (first_cross.ip <= 1e-3).mean() * 100
print(f"objects that ever cross 0.01%: {len(first_cross):,}")
print(f"first-crossing IP == exactly 1 draw (0.01%): {single:.1f}%")
print(f"first-crossing IP <= 0.1% (<=10 draws):      {low:.1f}%")
print(f"median first-crossing IP: {first_cross.ip.median()*100:.2f}%")

record("R6", "Fig 5's 0.01% 'high-confidence threshold' = a single MC draw", "(review concern)",
       f"{single:.1f}% of first crossings are exactly 1 impacting variant (median first-crossing "
       f"IP {first_cross.ip.median()*100:.2f}%)",
       "📊 COMPUTED",
       "Confirms the reviewer's concern is well-posed but most crossings jump straight past one draw; "
       "review decided to drop Fig 5 regardless")

objects that ever cross 0.01%: 19,318
first-crossing IP == exactly 1 draw (0.01%): 12.9%
first-crossing IP <= 0.1% (<=10 draws):      38.2%
median first-crossing IP: 0.30%

[R6] 📊 COMPUTED
  paper:    (review concern)
  computed: 12.9% of first crossings are exactly 1 impacting variant (median first-crossing IP 0.30%)
  note:     Confirms the reviewer's concern is well-posed but most crossings jump straight past one draw; review decided to drop Fig 5 regardless


## R7 — Fig 9 y-axis: PDF or count?

> Review: *"Fig 9 - Is PDF really right here on the Y axis? Seems like it should just be count"*
> Fig 9 caption already says *"The y-axis gives the number of impactors per bin (not a percentage)"* — but the axis label reads "PDF".

Exact replication of `plot_warning_time_histogram` (`plots.py:171`): cohort = complete + non-null discovery time; value = impact − 1%-crossing in years; **NaN → 0** (never-crossing objects are dumped into the first bin); `ax.hist` with default `density=False` → raw counts.

In [23]:
wt_fig9 = pc.subtract(cd.orbit.impact_time.mjd(),
                      cd.ip_threshold_1_percent.mjd()).to_numpy(zero_copy_only=False)
wt_fig9 = np.where(np.isnan(wt_fig9), 0, wt_fig9) / 365.25   # plots.py:203 NaN -> 0
bins = np.arange(0, np.ceil(np.nanmax(wt_fig9)), 1.0)
peaks = {}
for d in sorted(set(d_cd)):
    m = d_cd == d
    h, _ = np.histogram(wt_fig9[m], bins=bins)
    never = int(pc.is_null(cd.apply_mask(m).ip_threshold_1_percent.mjd())
                .to_numpy(zero_copy_only=False).sum())
    peaks[f"{d*1000:.0f}m"] = {"n objects": int(m.sum()), "first-bin count": int(h[0]),
                               "peak bin count": int(h.max()), "never-cross-1% in bin 0": never}
print(pd.DataFrame(peaks).T)

mx = max(v["peak bin count"] for v in peaks.values())
record("R7", "Fig 9 y-axis labeled 'PDF' but values are raw counts per bin", "y values up to ~300",
       f"replicating the plot code gives raw counts peaking at {mx} — the figure's scale. "
       f"y-axis is COUNT, not a PDF",
       "📝 WORDING",
       "Two fixes when regenerating: relabel y-axis to 'Count', AND note the NaN->0 line "
       "(plots.py:203) silently piles never-crossing discovered objects into the first bin — "
       "the 0-yr spike is partly an artifact")

       n objects  first-bin count  peak bin count  never-cross-1% in bin 0
40m          620              266             266                      156
80m         1372              313             313                      171
140m        2246              286             286                      147
250m        3153              231             231                      107
500m        4092              166             166                       67
1000m       4469              129             129                       57

[R7] 📝 WORDING
  paper:    y values up to ~300
  computed: replicating the plot code gives raw counts peaking at 313 — the figure's scale. y-axis is COUNT, not a PDF
  note:     Two fixes when regenerating: relabel y-axis to 'Count', AND note the NaN->0 line (plots.py:203) silently piles never-crossing discovered objects into the first bin — the 0-yr spike is partly an artifact


## R8 — Albedo model vs NEOMOD3

> Review: *"NEOMOD3 has pV,dark ≃ 0.03 and pV,bright ≃ 0.17 - i.e. there's a size and orbit dependence to albedo (as compared to our bimodal Rayleigh). We don't need to go back and adopt this, but we should at least note that we took a simplified model."*

What our bimodal Rayleigh actually produced, by taxonomic class:

In [24]:
alb = summary.orbit.albedo.to_numpy(zero_copy_only=False)
ast = np.array(summary.orbit.ast_class.to_pylist())
for cls in np.unique(ast):
    m = ast == cls
    sigma = np.sqrt(np.mean(alb[m] ** 2) / 2)   # Rayleigh MLE; mode = sigma
    print(f"{cls}-type: n={m.sum():,}  median {np.median(alb[m]):.3f}  "
          f"mode(sigma) {sigma:.3f}  mean {alb[m].mean():.3f}")

sig_c = np.sqrt(np.mean(alb[ast == 'C'] ** 2) / 2)
sig_s = np.sqrt(np.mean(alb[ast == 'S'] ** 2) / 2)
record("R8", "Albedo model comparison vs NEOMOD3 (pV dark 0.03 / bright 0.17)", "0.03 / 0.17 (NEOMOD3)",
       f"our Rayleigh modes: C {sig_c:.3f} / S {sig_s:.3f}",
       "📊 COMPUTED",
       "For the suggested 'simplified model' caveat: no size or orbit dependence in our draw")

C-type: n=7,100  median 0.034  mode(sigma) 0.029  mean 0.036
S-type: n=22,900  median 0.195  mode(sigma) 0.168  mean 0.208

[R8] 📊 COMPUTED
  paper:    0.03 / 0.17 (NEOMOD3)
  computed: our Rayleigh modes: C 0.029 / S 0.168
  note:     For the suggested 'simplified model' caveat: no size or orbit dependence in our draw


---
# Adopted cohort: global prograde filter (decision 2026-07-08)

The retrograde tail (1,962 objects, 6.5%, clustered at i ≈ 180° — median exactly 180.0°) is an
artifact of the iterative impact-enforcement step: the real NEO population and the Granvik source
model contain essentially no retrograde objects. **Decision: exclude i ≥ 90° globally.**

Cohort conventions going forward:
- **All stats**: prograde only (i < 90°), N = 28,038
- **Discovery stats**: raw non-null discovery time (incomplete runs included — discovery depends
  only on the Sorcha observations, not the failed OD/IP pipeline)
- **IP-dependent stats**: additionally require `status == "complete"`

This section recomputes every paper number under the adopted cohort.

## P1 — Headline statistics under the global prograde cohort

In [25]:
prog_np = summary.prograde_orbits().to_numpy(zero_copy_only=False)
imp_mjd = summary.orbit.impact_time.mjd().to_numpy(zero_copy_only=False)
t1_mjd = summary.ip_threshold_1_percent.mjd().to_numpy(zero_copy_only=False)
t90_mjd = summary.ip_threshold_90_percent.mjd().to_numpy(zero_copy_only=False)
dt_mjd = summary.discovery_time.mjd().to_numpy(zero_copy_only=False)
opt_mjd = summary.discovery_time_optimistic.mjd().to_numpy(zero_copy_only=False)
ip_disc_arr = summary.ip_at_discovery_time.to_numpy(zero_copy_only=False)

print(f"prograde cohort: {prog_np.sum():,} of 30,000 ({prog_np.mean()*100:.1f}%)")
combo_p = collections.Counter(
    (d, dec) for d, dec, p in zip(diam_km, decades, prog_np) if p)
print(f"per-(decade,diameter) counts now range {min(combo_p.values())}-{max(combo_p.values())} "
      f"(was uniformly 500)")

# --- discovery rates (raw discovery, prograde) ---
rates_p = {d: disc_mask[prog_np & (diam_km == d)].mean() * 100 for d in sorted(set(diam_km))}
straight_p = np.mean([v for k, v in rates_p.items() if k >= 0.14])
small_p = disc_mask[prog_np & (diam_km < 0.14)].mean() * 100
edges = np.array([0.14, 0.25, 0.5, 1.0, 2.0])
wts = edges[:-1] ** -2.35 - edges[1:] ** -2.35
sfd_p = np.average([rates_p[d] for d in (0.14, 0.25, 0.5, 1.0)], weights=wts)
# Empirically anchored SFD: N(>140m) ~ 25,000 and N(>1km) ~ 940 -> cumulative slope 1.67.
# Recommended number to quote: 62% of the real >=140m population is in the 140-250m bin.
b_emp = np.log(25000 / 940) / np.log(1000 / 140)
Ncum = 25000 * (edges / 0.14) ** -b_emp
w_emp = Ncum[:-1] - Ncum[1:]
w_emp[-1] = Ncum[-2]
sfd_emp = np.average([rates_p[d] for d in (0.14, 0.25, 0.5, 1.0)], weights=w_emp)

# --- IP stats (prograde + complete + discovered) ---
pcd = prog_np & complete_np & disc_mask
ip_v = ip_disc_arr[pcd]; ip_v = ip_v[~np.isnan(ip_v)]
med_ip_all = np.median(ip_v) * 100
med_ip_nz = np.median(ip_v[ip_v > 0]) * 100
zero_frac = (ip_v == 0).mean() * 100
reach90_p = (~np.isnan(t90_mjd[pcd])).mean() * 100
never1_p = np.isnan(t1_mjd[pcd]).mean() * 100

# --- discovery->1% crossing (discovered below 1%) ---
below = pcd & (np.nan_to_num(ip_disc_arr, nan=0.0) < 0.01)
d2one = t1_mjd - dt_mjd
any_cross = below & ~np.isnan(d2one)
med_45, mean_326 = np.median(d2one[any_cross]), np.mean(d2one[any_cross])

# --- warning times (prograde + complete + discovered; types.py:257 definition) ---
wt_p = np.maximum(imp_mjd - t1_mjd, imp_mjd - dt_mjd) / 365.25
w140_2036 = np.nanmean(wt_p[pcd & (diam_km == 0.14) & (iy == 2036)])
w1km_2045 = np.nanmean(wt_p[pcd & (diam_km == 1.0) & (iy == 2045)])
w40_2045 = np.nanmean(wt_p[pcd & (diam_km == 0.04) & (iy == 2045)])

# --- linking (already prograde from C7/C14/C15) ---
n_obnd = len(obnd_prog)
n_rec = int((~pc.is_null(obnd_prog.discovery_time_optimistic.mjd())
             .to_numpy(zero_copy_only=False)).sum())
gains = (dt_mjd - opt_mjd)[pcd]
gains = gains[~np.isnan(gains)]

comparison = pd.DataFrame([
    ("population N",                    "30,000",        f"{prog_np.sum():,}"),
    (">=140m discovery (bin mean)",     "70.4%",         f"{straight_p:.1f}%"),
    ("  per-bin 40/80/140/250/500/1000","12.4/27.5/45.1/63.5/82.6/90.6",
     "/".join(f"{rates_p[d]:.1f}" for d in sorted(rates_p))),
    ("  SFD-weighted (alpha=2.35)",     "~50%",          f"{sfd_p:.1f}%"),
    ("  SFD-weighted (empirical anchor)","~54%",         f"{sfd_emp:.1f}%"),
    ("<140m discovery (pooled)",        "20.0%",         f"{small_p:.1f}%"),
    ("median IP at discovery (all)",    "2.9%",          f"{med_ip_all:.1f}%"),
    ("median IP at discovery (nonzero)","6.5%",          f"{med_ip_nz:.1f}%"),
    ("  IP=0 at discovery",             "12%",           f"{zero_frac:.0f}%"),
    ("discovered reaching IP>90%",      "83.0%",         f"{reach90_p:.1f}%"),
    ("discovered never reaching 1%",    "4.4%",          f"{never1_p:.1f}%"),
    ("disc->1% cross median/mean",      "47 d / 316 d",  f"{med_45:.0f} d / {mean_326:.0f} d"),
    ("warning 140m impact-2036",        "8.0 yr",        f"{w140_2036:.1f} yr"),
    ("warning 1km impact-2045",         "18.1 yr",       f"{w1km_2045:.1f} yr"),
    ("warning 40m impact-2045",         "12.7 yr",       f"{w40_2045:.1f} yr"),
    ("observed-but-not-discovered",     "6,102",         f"{n_obnd:,}"),
    ("recoverable via linking",         "2,365 (38.8%)", f"{n_rec:,} ({n_rec/n_obnd*100:.1f}%)"),
    ("linking gain median/mean",        "+8 d / +242 d", f"+{np.median(gains):.0f} d / +{np.mean(gains):.0f} d"),
], columns=["stat", "full cohort (paper-era)", "PROGRADE (adopted)"])
print(comparison.to_string(index=False))

record("P1", "All headline stats recomputed under the adopted global prograde cohort",
       "(see table)", "table above", "🔁 RECOMPUTED",
       "Retrograde exclusion mostly nudges discovery rates up ~1 pp; linking and IP stats "
       "essentially unchanged")

prograde cohort: 28,038 of 30,000 (93.5%)
per-(decade,diameter) counts now range 457-480 (was uniformly 500)
                             stat       full cohort (paper-era)            PROGRADE (adopted)
                     population N                        30,000                        28,038
      >=140m discovery (bin mean)                         70.4%                         71.5%
   per-bin 40/80/140/250/500/1000 12.4/27.5/45.1/63.5/82.6/90.6 13.3/29.3/47.1/65.1/83.4/90.5
        SFD-weighted (alpha=2.35)                          ~50%                         52.6%
  SFD-weighted (empirical anchor)                          ~54%                         56.4%
         <140m discovery (pooled)                         20.0%                         21.3%
     median IP at discovery (all)                          2.9%                          2.9%
 median IP at discovery (nonzero)                          6.5%                          6.5%
                IP=0 at discovery            

## P2 — Figure 8 warning-time grid, prograde cohort

In [26]:
rows_p = {}
for year in range(2036, 2046):
    rows_p[year] = {f"{d*1000:.0f}m": np.nanmean(wt_p[pcd & (diam_km == d) & (iy == year)])
                    for d in sorted(set(diam_km))}
fig8_p = pd.DataFrame(rows_p).T
print("Mean warning time (years), prograde cohort — new Figure 8 values:")
print(fig8_p.round(1))
record("P2", "Figure 8 grid under prograde cohort", "(see C12 for old grid)",
       "table above", "🔁 RECOMPUTED")

Mean warning time (years), prograde cohort — new Figure 8 values:
       40m   80m  140m  250m  500m  1000m
2036   1.7   6.9   8.0   8.4   9.3    9.3
2037   7.5   7.6   8.8   8.9   9.4    9.8
2038   8.7   9.8  10.1  10.4  11.0   11.4
2039  11.2  11.2  11.6  11.4  12.4   12.5
2040  11.5  11.8  11.6  12.5  13.0   13.4
2041  12.3  12.5  13.0  13.4  13.5   14.0
2042  15.6  13.3  14.2  14.4  14.8   15.4
2043  15.6  13.8  14.9  15.1  15.9   16.1
2044  13.7  15.8  15.0  16.9  17.2   17.3
2045  12.7  13.9  15.9  16.9  18.0   18.0

[P2] 🔁 RECOMPUTED
  paper:    (see C12 for old grid)
  computed: table above


## P3 — Orbit-category breakdown (R1/R2), prograde cohort

In [27]:
big = diam_km >= 0.14
groups_p = {
    "prograde >=140m population":   prog_np & big,
    "  discovered":                 prog_np & big & complete_np & disc_mask,
    "  undiscovered (all)":         prog_np & big & complete_np & ~disc_mask,
    "    observed-not-discovered":  prog_np & big & complete_np & ~disc_mask & (obs_np > 0),
    "    never observed":           prog_np & big & complete_np & ~disc_mask & (obs_np == 0),
    "  incomplete runs (excluded)": prog_np & big & ~complete_np,
}
rows = []
for label, m in groups_p.items():
    n = m.sum()
    inner = int((a_au[m] < 1).sum())
    band = int((np.abs(a_au[m] - 1) < 0.05).sum())
    rows.append({"group": label, "n": n,
                 "a<1AU": inner, "a<1AU %": inner / n * 100,
                 "|a-1|<0.05": band, "|a-1|<0.05 %": band / n * 100})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.1f}"))

pdisc = prog_np & big & disc_mask
print(f"\ndiscovery rate >=140m prograde: a<1AU "
      f"{disc_mask[prog_np & big & (a_au < 1)].mean()*100:.1f}% vs a>=1AU "
      f"{disc_mask[prog_np & big & (a_au >= 1)].mean()*100:.1f}%; band "
      f"{disc_mask[prog_np & big & (np.abs(a_au-1) < 0.05)].mean()*100:.1f}%")

record("P3", "R1/R2 orbit-category numbers under prograde cohort", "(see R1/R2)",
       "table above", "🔁 RECOMPUTED")

                       group     n  a<1AU  a<1AU %  |a-1|<0.05  |a-1|<0.05 %
  prograde >=140m population 18692   3804     20.4        1236           6.6
                  discovered 13243   2321     17.5         705           5.3
          undiscovered (all)  5310   1468     27.6         527           9.9
     observed-not-discovered  3543   1143     32.3         234           6.6
              never observed  1767    325     18.4         293          16.6
  incomplete runs (excluded)   139     15     10.8           4           2.9

discovery rate >=140m prograde: a<1AU 61.3% vs a>=1AU 74.1%; band 57.3%

[P3] 🔁 RECOMPUTED
  paper:    (see R1/R2)
  computed: table above


---
# Retrograde-artifact diagnosis (impactor-generation code, 2026-07-13)

Source: `B612-Asteroid-Institute/impactor-generation` @ 5281fa3, `notebooks/KK_generateimpactors.ipynb`
+ `granvikneos/` (Fortran sampler). Pipeline: draw 600k NEOs from the Granvik lores model →
filter to `LMA` (nodal miss distance < gravitationally-focused Earth radius) → **sample 20,000
with replacement weighted by `chesley_prob`** → `create_impact` minimizes Earth-miss distance
with a, e, i free within ±2% (plus w, node, M, t free) → study replicates each orbit ×6 diameters.

**Mechanism (three compounding steps):**
1. **Source tail**: the Granvik lores model has small nonzero weight in high-i bins; the sampler
   smears bin centers ±2°, yielding rare candidates at i ≈ 90–180°.
2. **Selection amplification**: the ω-acceptance of the LMA cut scales with plane proximity
   (dmin ∝ cos φ → 0 as i→0° or →180°) — this favors near-plane draws of EITHER direction; at
   matched plane distance retrograde acceptance is actually ~35% LOWER (U ≈ 68 km/s → no
   gravitational focusing → smaller target). What is intrinsically retrograde is the WEIGHT:
   sinθ = Ux/U tiny for head-on, Earth-grazing (q≈1) encounters → `chesley_prob` ∝ 1/sinθ ≈ 18
   (vs ~1-2.5 prograde). Sampling 20k **with replacement** (fixed `random_state=1111`) then
   duplicates the few high-weight candidates dozens of times.
3. **Enforcement pinning**: `create_impact` — unlike the repo's original `find_impact_*`
   functions, which hold a, e, i **fixed** — lets the optimizer move i by ±2% (±3.5° at
   i≈175°). Earth lies in the ecliptic, so the distance minimum is at the in-plane boundary:
   retrogrades pile up at exactly 179.5–180° (i>π solutions fold back), and prograde low-i
   orbits pile up toward 0°.

The prograde filter removes the retrograde artifact but NOT the mirrored i≈0 pile-up
(15–25× excess below 0.25°), which sits exactly in the paper's "hard to discover" population.

## G1 — Anatomy of the retrograde population

In [28]:
kep_all = summary.orbit.coordinates.to_keplerian()
ga = kep_all.a.to_numpy(zero_copy_only=False)
ge = kep_all.e.to_numpy(zero_copy_only=False)
gi = kep_all.i.to_numpy(zero_copy_only=False)
gdec = np.array(decades)

retro_m = gi >= 90
print(f"retrograde rows: {retro_m.sum():,} of 30,000 ({retro_m.mean()*100:.2f}%)")
print("\ninclination distribution of retrogrades (deg):")
print(pd.cut(gi[retro_m], [90, 150, 170, 175, 178, 179.5, 180]).value_counts().sort_index())

# every decade holds 500 distinct orbits x 6 diameter bins
one = pd.DataFrame({"a": ga, "e": ge, "i": gi, "dec": gdec})
n_distinct = one[one.dec == "2025-2035"].groupby(
    [one.a.round(6), one.e.round(6), one.i.round(6)]).ngroups
print(f"\ndistinct orbits in decade 2025-2035: {n_distinct} (x6 diameters = 3,000 rows)")

rd = one[retro_m]
distinct_retro = rd.groupby([rd.a.round(6), rd.e.round(6)]).ngroups
print(f"distinct retrograde orbits across all decades: {distinct_retro} "
      f"({distinct_retro}/5,000 = {distinct_retro/5000*100:.1f}% of distinct orbits)")

# duplication clusters (same generator candidate resampled with replacement)
rd2 = rd.copy()
rd2["key"] = rd2.a.round(2).astype(str) + "_" + rd2.e.round(2).astype(str)
vc = rd2.key.value_counts()
print(f"\n(a,e)-clusters among retrogrades: {len(vc)}; top sizes {vc.head(5).tolist()}")
print("top clusters all sit at a=3.0-3.1, e=0.68-0.75, i=180 (q=0.75-1.0 au) — "
      "a single JFC-like Granvik bin region, massively resampled")

# the mirrored artifact at i=0 (NOT removed by the prograde filter)
low = gi[gi < 5]
for cut in (0.1, 0.25):
    expected = len(low) * (cut / 5) ** 2   # Harris quadratic fix: pdf ~ i -> CDF ~ i^2
    observed = (low < cut).sum()
    print(f"i < {cut} deg: observed {observed}, Granvik-expected ~{expected:.0f} "
          f"({observed/expected:.0f}x excess)")

record("G1", "Retrograde population is a generator artifact (selection amplification + "
       "enforcement pinning of a tiny Granvik high-i tail)",
       "(diagnosis)", f"{distinct_retro} distinct orbits -> {retro_m.sum():,} rows; 84% pinned "
       f"at 179.5-180 deg; mirrored 15-25x excess at i<0.25 deg remains in prograde cohort",
       "🔬 DIAGNOSED",
       "Fixes for a rerun: hold i fixed in create_impact (as in original find_impact_*), or "
       "clip i to [0, 180] and cut retrograde draws; consider capping resampling weights")

retrograde rows: 1,962 of 30,000 (6.54%)

inclination distribution of retrogrades (deg):
(90.0, 150.0]      156
(150.0, 170.0]      78
(170.0, 175.0]       6
(175.0, 178.0]      48
(178.0, 179.5]      24
(179.5, 180.0]    1650
Name: count, dtype: int64

distinct orbits in decade 2025-2035: 500 (x6 diameters = 3,000 rows)
distinct retrograde orbits across all decades: 326 (326/5,000 = 6.5% of distinct orbits)

(a,e)-clusters among retrogrades: 256; top sizes [36, 30, 24, 24, 18]
top clusters all sit at a=3.0-3.1, e=0.68-0.75, i=180 (q=0.75-1.0 au) — a single JFC-like Granvik bin region, massively resampled
i < 0.1 deg: observed 114, Granvik-expected ~4 (25x excess)
i < 0.25 deg: observed 420, Granvik-expected ~28 (15x excess)

[G1] 🔬 DIAGNOSED
  paper:    (diagnosis)
  computed: 326 distinct orbits -> 1,962 rows; 84% pinned at 179.5-180 deg; mirrored 15-25x excess at i<0.25 deg remains in prograde cohort
  note:     Fixes for a rerun: hold i fixed in create_impact (as in original find_i

## P4 — Final cohort audit (2026-07-16)

**Adopted convention:** discovery statistics = prograde, including runs that finished Sorcha but
not the OD/IP pipeline (`status == "incomplete"`); ALL other statistics = prograde + complete.

This cell verifies (a) that including incomplete runs in discovery stats is justified, (b) that
every headline number and figure is consistent with the convention, and (c) the size of every
residual cohort discrepancy.

In [29]:
# (a) All incomplete runs finished Sorcha -> discovery status is knowable
inc = summary.apply_mask(summary.incomplete())
inc_obs = inc.observations.to_numpy(zero_copy_only=False)
print(f"incomplete runs: {len(inc)}; all have observations > 0: {(inc_obs > 0).all()}")

# (b) headline discovery: adopted (incl. incomplete) vs complete-only (what Fig 1 uses internally)
for label, m in [("ADOPTED (incl. incomplete)", prog_np),
                 ("complete-only (Fig 1 internal)", prog_np & complete_np)]:
    rates = [disc_mask[m & (diam_km == d)].mean() * 100 for d in (0.14, 0.25, 0.5, 1.0)]
    print(f"  {label}: >=140m mean {np.mean(rates):.2f}%")

# (c) residual discrepancies, quantified
n_inc_disc = (prog_np & ~complete_np & disc_mask).sum()
n_inc_obnd = (prog_np & ~complete_np & (obs_np > 0) & ~disc_mask).sum()
print(f"\nprograde incomplete discovered (in headline, not in Fig 1 bars): {n_inc_disc}")
print(f"prograde incomplete observed-not-discovered (excluded from 6,102): {n_inc_obnd}")

audit = pd.DataFrame([
    ("headline discovery rates (P1)",  "prograde, incl. incomplete",  "✅ matches convention"),
    ("IP/warning/linking stats (P1)",  "prograde + complete",         "✅ matches convention"),
    ("Fig 1 discovered-by-window",     "prograde + complete (internal filter; needed for IP shading)",
     "✅ within rounding: 71.44% vs 71.51% headline — both print 71.5%"),
    ("Fig 2 elements 3-category",      "prograde + complete",         "✅ acceptable (139 excluded)"),
    ("Fig 3 max IP by decade",         "prograde + complete + discovered", "✅ correct (IP-dependent)"),
    ("Fig 4 IAWN not reached",         "prograde + complete + discovered", "✅ correct (IP-dependent)"),
    ("Fig 6 arc length by decade",     "prograde + complete",         "✅ acceptable (obs-based; 139 excluded)"),
    ("Fig 8 warning by year",          "prograde + complete + discovered", "✅ correct (IP-dependent)"),
    ("Fig 9 warning histogram",        "prograde + complete + crossed 1%", "✅ correct (IP-dependent)"),
    ("6,102 obs-not-discovered (3.5)", "prograde + complete",
     "📝 definitional: 17 incomplete obs-not-disc excluded — state 'complete runs' in text"),
], columns=["item", "cohort used", "verdict"])
with pd.option_context("display.max_colwidth", 75, "display.width", 220):
    print("\n" + audit.to_string(index=False))

record("P4", "Final cohort audit: all numbers and figures vs adopted convention",
       "(consistency check)",
       f"all consistent; max residual = 0.07 pp on headline (rounding-safe); "
       f"{n_inc_obnd} incomplete obs-not-disc excluded from 6,102 by design",
       "✅ VERIFIED",
       "Paper should state: 'discovery statistics include the 147 runs (0.5%) whose synthetic "
       "observations completed but whose OD/IP pipeline did not; all IP-dependent statistics "
       "exclude them'")

incomplete runs: 147; all have observations > 0: True
  ADOPTED (incl. incomplete): >=140m mean 71.51%
  complete-only (Fig 1 internal): >=140m mean 71.44%

prograde incomplete discovered (in headline, not in Fig 1 bars): 128
prograde incomplete observed-not-discovered (excluded from 6,102): 17

                          item                                                  cohort used                                                                             verdict
 headline discovery rates (P1)                                   prograde, incl. incomplete                                                                ✅ matches convention
 IP/warning/linking stats (P1)                                          prograde + complete                                                                ✅ matches convention
    Fig 1 discovered-by-window prograde + complete (internal filter; needed for IP shading)                     ✅ within rounding: 71.44% vs 71.51% headline — both print 71.5%

## Summary of verdicts

In [30]:
vt = pd.DataFrame(VERDICTS)[["id", "verdict", "paper", "computed", "claim"]]
with pd.option_context("display.max_colwidth", 90, "display.width", 250):
    print(vt.to_string(index=False))
print()
for v in VERDICTS:
    if v["note"]:
        print(f"[{v['id']}] {v['note']}")

  id      verdict                                   paper                                                                                                                        computed                                                                                                                       claim
  C1   ✅ VERIFIED            30,000; 500/combo; 2025-2125                                                                                                    30,000; 500/combo; 2025-2125                                                    30,000 impactors = 500 x 10 decades x 6 diameter bins, impacts 2025-2125
  C2   ✅ VERIFIED                                  10,000                                                                                               IP quantum = 0.0001 -> N = 10,000                                                                      10,000 Monte Carlo variants per nightly IP calculation
  C3   ✅ VERIFIED                            min 3 nights             

## Claims not verifiable from the archived data (config / literature)

| Claim | Source | Status |
|---|---|---|
| `baseline_v4.3.1_10yrs` pointing database | §2.2 | Config — pointing DB not archived alongside `mar_run`; consistent with population file names (`...4.3.1...params`) |
| MOPS discovery criterion: 3 × 2-obs tracklets in ≤15 nights | §2.3 | Implemented in `compute_discovery_dates` (`main.py:34`) with `min_tracklets=3, max_nights=15`; note the window mask `night >= n-15` spans 16 night labels inclusive — borderline off-by-one vs "at most 15 nights" |
| Perfect precovery / perfect linking assumptions | §2.3, §4.4 | Stated assumptions, not data claims |
| Jones et al. (2018) PHA discovery rate 65.6% | §3.1 | Literature value — but see C4: the comparison is undermined if 70.4% is an unweighted average |
| Rubin facts (8.4 m mirror, 9.6 deg² FoV, r≈24.5, ~130 NEOs/night, >60% of ≥140 m NEOs) | §1 | Literature (Ivezić 2019, Jones 2018) |
| Population method: Granvik et al. (2018) model, Chesley et al. (2024) approach, bimodal Rayleigh albedos (Thomas et al. 2011) | §2.1 | Methodology provenance; input files in `data/inputs/12-02-2024/` |
| Software: Sorcha, Find Orb, ASSIST, ADAM Core | §2, Software section | Citations |
| ASSIST "all major solar system bodies" | §2.4 | Verified in the study venv (adam-assist 0.3.0, default forces): JPL **DE440** planets ephemeris + **SB441-N16** — Sun, 8 planets, Moon, Pluto, and 16 massive asteroids (Ceres, Pallas, Juno, Vesta, Iris, Hygiea, Eunomia, Psyche, Euphrosyne, Europa, Cybele, Sylvia, Thisbe, Camilla, Davida, Interamnia) — plus Earth J2/J3/J4, solar J2, and EIH relativistic corrections. NON_GRAVITATIONAL force is enabled but inert (variants carry no A1/A2/A3): propagation is gravity-only. |

## Editorial flags (draft issues spotted while verifying)

1. **§3.3 dangling sentence** — *"More precisely, 83.0% of discovered objects reach an IP exceeding 90% before the end of the 10-year survey. **The 17**"* — sentence cuts off mid-thought.
2. **C4 wording** — "averaged over the realistic distribution of sizes" describes a weighting the code does not perform (see C4).
3. **Prograde filter unmentioned** — several paper numbers (6,102 among them) require the i<90° filter; the population section never mentions excluding retrograde objects (1,962 objects, 6.5% of the population).
4. **Median IP definition** — the 7% median at discovery only emerges after excluding IP=0 discoveries (see C8).
5. **Warning-time definition** — text defines warning time from the 1%-crossing epoch, code uses `max(crossing, discovery)` (`types.py:257`); same number for most objects but not all.
6. **Realization-time definition** — "first detection" and "permanently exceeds" vs code's discovery-to-first-crossing (see C13).
7. **Figure 1 axis** — only shows impact windows through 2065–2069 though the data runs to 2125 (matches the `_2070` variant of the archived figure).